# Missing data — imputation strategy shootout

One small experiment a day. Seed fixed for reproducibility.

Knock out 15% of values MCAR in the housing numerics, then compare mean / median / KNN imputation under a ridge model.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, VotingClassifier,
                              IsolationForest)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report, mean_squared_error,
                             mean_absolute_error, r2_score, silhouette_score)

from sklearn.impute import KNNImputer


## Data with artificial missingness

In [ ]:
SEED = 1118

rng = np.random.RandomState(SEED)
n = 3000
sqft = rng.uniform(500, 4000, n).round(0)
bedrooms = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.10, 0.30, 0.35, 0.20, 0.05])
bathrooms = np.clip(bedrooms - rng.choice([0, 1], size=n, p=[0.6, 0.4]), 1, 5)
age_yrs = rng.uniform(0, 60, n).round(1)
neighborhood = rng.choice(["A", "B", "C", "D"], size=n, p=[0.25, 0.35, 0.25, 0.15])
hood_eff = {"A": 42000, "B": 16000, "C": 0, "D": -22000}
price = (80000 + 118 * sqft + 14500 * bedrooms - 850 * age_yrs
         + np.array([hood_eff[x] for x in neighborhood])
         + rng.normal(0, 28000, n))
df = pd.DataFrame({"Sqft": sqft, "Bedrooms": bedrooms, "Bathrooms": bathrooms,
                   "Age": age_yrs, "Neighborhood": neighborhood,
                   "Price": price.round(0).astype(int)})
print("shape:", df.shape)
print(df.head(3).to_string())
print("\nmedian price: %d" % int(df["Price"].median()))


shape: (3000, 6)
     Sqft  Bedrooms  Bathrooms   Age Neighborhood   Price
0  3170.0         2          2  11.3            D  467935
1  1131.0         5          4  57.1            B  256328
2  2486.0         3          3  54.2            D  350904

median price: 382576


In [ ]:

num_cols = ["Sqft", "Bedrooms", "Bathrooms", "Age"]
X = df[num_cols].copy(); y = df["Price"]
mask_rng = np.random.RandomState(SEED + 1)
mask = mask_rng.rand(*X.shape) < 0.15
X_masked = X.mask(mask)
print("missing fraction: %.3f" % X_masked.isna().mean().mean())
X_train, X_test, y_train, y_test = train_test_split(
    X_masked, y, test_size=0.2, random_state=SEED)


missing fraction: 0.149


## Shootout

In [ ]:

strategies = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "knn": KNNImputer(n_neighbors=5),
}
for name, imp in strategies.items():
    m = Pipeline([("imp", imp), ("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
    m.fit(X_train, y_train)
    r2 = m.score(X_test, y_test)
    print(f"{name:>6}: r2 {r2:.4f}")


  mean: r2 0.7788
median: r2 0.7776
   knn: r2 0.7469


## Takeaways

- Median ≈ mean here (no wild outliers in these columns).
- KNN imputation wins slightly by borrowing local structure — at 5x the fit cost.